# 1. Data Loading and Interaction Matrix Construction

- start by loading the playlist and track data, then build an interaction matrix where each row represents a playlist and each column represents a unique track. A cell is set to 1 if the track is in the playlist.

In [21]:
import pandas as pd
import numpy as np

# Load playlists and tracks
playlists = pd.read_parquet('/Users/xavierhua/Documents/GitHub/spotifynd/phase3_feature_engineering/feature engineered datasets/playlist.parquet')
tracks = pd.read_parquet('/Users/xavierhua/Documents/GitHub/spotifynd/phase3_feature_engineering/feature engineered datasets/tracks_temp.parquet')

# Filter playlists to include only those with at least 20 tracks
playlists = playlists[playlists['track_idx_list'].apply(len) >= 20].reset_index(drop=True)[:1000]

# Extract playlist IDs and corresponding track lists
playlist_ids = playlists['playlist_idx'].tolist()
playlist_tracks = dict(zip(playlists['playlist_idx'], playlists['track_idx_list']))

# Create a sorted list of all unique track IDs
unique_tracks = set()
for tlist in playlist_tracks.values():
    unique_tracks.update(tlist)
unique_tracks = sorted(list(unique_tracks))

# Map each track ID to a column index in the interaction matrix
track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}

# Initialize the interaction matrix
interaction_matrix = np.zeros((len(playlist_ids), len(unique_tracks)))

# Fill in the matrix: 1 if the track is in the playlist, 0 otherwise
for i, pid in enumerate(playlist_ids):
    for track in playlist_tracks[pid]:
        j = track_to_col[track]
        interaction_matrix[i, j] = 1  # use count if needed instead of binary 1

print("Filtered playlists:", len(playlist_ids))
print("Unique tracks:", len(unique_tracks))
print("Interaction matrix shape:", interaction_matrix.shape)

Filtered playlists: 1000
Unique tracks: 38732
Interaction matrix shape: (1000, 38732)


# 2. Collaborative Filtering Methods

- We implement three different recommendation methods.



### 2.1. Item-Based Collaborative Filtering
- For item-based CF, we compute similarities between tracks based on the playlists in which they appear. For a given track, find similar tracks that tend to co-occur in playlists, then recommend those.

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_item_similarity(train_matrix):
    """Compute cosine similarity between tracks (columns)."""
    return cosine_similarity(train_matrix.T)

def predict_item_based(playlist_idx, train_matrix, sim_matrix=None):
    """
    Predict scores for a given playlist using item-based CF.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The interaction matrix.
      - sim_matrix: Precomputed item similarity matrix (optional).
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    playlist_vector = train_matrix[playlist_idx].reshape(1, -1)
    
    # Use precomputed similarity matrix if available
    if sim_matrix is not None:
        predicted_scores = playlist_vector.dot(sim_matrix)
    else:
        sim = cosine_similarity(train_matrix.T)
        predicted_scores = playlist_vector.dot(sim)
    
    # Remove tracks already in the playlist
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[0, existing_track_indices] = -np.inf
    
    return predicted_scores.flatten()

### 2.2. User-Based Collaborative Filtering

- For user-based CF, we compute similarities between playlists. For a given playlist, find other similar playlists and recommend tracks that are present in these similar playlists but missing in the current one.

In [23]:
def compute_user_similarity(train_matrix):
    """Compute cosine similarity between playlists (rows)."""
    return cosine_similarity(train_matrix)

def predict_user_based(playlist_idx, train_matrix, user_sim_matrix=None):
    """
    Predict scores for a given playlist using user-based CF.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The interaction matrix.
      - user_sim_matrix: Precomputed user similarity matrix (optional).
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    if user_sim_matrix is None:
        user_sim_matrix = compute_user_similarity(train_matrix)
    
    sim_scores = user_sim_matrix[playlist_idx]
    predicted_scores = sim_scores.dot(train_matrix)
    
    # Remove tracks already present in the playlist
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

### 2.3. Matrix Factorization using Truncated SVD
- We use truncated SVD to factorize the interaction matrix into latent factors. Then, recommendations are generated by computing the dot product of the latent factors.

In [24]:
from sklearn.decomposition import TruncatedSVD

# Choose the latent dimension (number of factors)
latent_dim = 100

# Perform truncated SVD
svd = TruncatedSVD(n_components=latent_dim, random_state=42)
U = svd.fit_transform(interaction_matrix)      # Shape: (num_playlists, latent_dim)
Sigma = svd.singular_values_                     # Shape: (latent_dim,)
VT = svd.components_                             # Shape: (latent_dim, num_tracks)

# Rescale using square root of singular values
sqrt_sigma = np.sqrt(Sigma)

# Obtain latent factors P (for playlists) and Q (for tracks)
P = U * sqrt_sigma         # Shape: (num_playlists, latent_dim)
Q = (VT.T * sqrt_sigma)    # Shape: (num_tracks, latent_dim)

print("P shape (playlist factors):", P.shape)
print("Q shape (track factors):", Q.shape)

def predict_mf(playlist_idx, train_matrix, P, Q):
    """
    Predict scores using matrix factorization latent factors.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: Interaction matrix (used for filtering).
      - P: Playlist latent factor matrix.
      - Q: Track latent factor matrix.
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    predicted_scores = P[playlist_idx].dot(Q.T)
    
    # Remove tracks already in the playlist
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

P shape (playlist factors): (1000, 100)
Q shape (track factors): (38732, 100)


### 2.4. Matrix Factorization using ALS
- Use ALS on a sparse implicit feedback matrix to learn latent factors for playlists and tracks, then generate recommendations by computing the dot product of these factors.

In [25]:
import numpy as np
from scipy.sparse import csr_matrix
import implicit

# Assume interaction_matrix is already created (dense NumPy array from previous code).
# Convert the dense interaction matrix into a sparse CSR matrix.
sparse_interaction = csr_matrix(interaction_matrix)

# Set parameters for ALS
latent_dim = 200  # You can experiment with different dimensions
regularization = 0.1
iterations = 20
alpha = 40  # Confidence scaling factor for implicit feedback

# Scale the data by confidence: higher weight for observed interactions.
data_conf = (sparse_interaction * alpha).astype('double')

# Initialize and train the ALS model.
als_model = implicit.als.AlternatingLeastSquares(factors=latent_dim,
                                                 regularization=regularization,
                                                 iterations=iterations,
                                                 random_state=42)
# Fit the model using the weighted implicit feedback.
als_model.fit(data_conf)

# Extract latent factors for playlists and tracks.
# Note: In the implicit package, 'user_factors' correspond to the rows of the sparse matrix
# and 'item_factors' correspond to its columns.
P_als = als_model.user_factors   # shape: (num_playlists, latent_dim)
Q_als = als_model.item_factors   # shape: (num_tracks, latent_dim)

print("P_als shape (playlist factors):", P_als.shape)
print("Q_als shape (track factors):", Q_als.shape)

def predict_als(playlist_idx, train_matrix, P_als, Q_als):
    """
    Predict scores using ALS latent factors.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The original dense interaction matrix (used for filtering).
      - P_als: Playlist latent factor matrix from ALS.
      - Q_als: Track latent factor matrix from ALS.
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    # Compute predicted scores as the dot product of the playlist's latent factors and all track factors.
    predicted_scores = P_als[playlist_idx].dot(Q_als.T)
    
    # Filter out tracks already in the playlist.
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

  0%|          | 0/20 [00:00<?, ?it/s]

P_als shape (playlist factors): (1000, 200)
Q_als shape (track factors): (38732, 200)


# 3. Evaluation

We evaluate our recommendation models using a **leave-k-out strategy**, where 10 tracks are removed from each playlist and used as test items. We compute the following metrics:

### 1. Hit Ratio @ K
The fraction of playlists where **at least one held-out track** appears in the top K recommendations. A higher value means a greater fraction of playlists have a relevant item in the top-K.

$$
HR@K = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}\{\text{any test item in top-}K \text{ for playlist } i\}
$$

---

### 2. Mean Reciprocal Rank (MRR)
The average of the reciprocal ranks of the **first relevant** (held-out) track across playlists. A higher MRR means the relevant item tends to appear closer to the top of the ranked list.

$$
MRR = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{\text{rank}_{i}^{(1)}}
$$

Where $\text{rank}_{i}^{(1)}$ is the rank of the first relevant (held-out) item for playlist \(i\).

---

### 3. Mean Average Precision (MAP) @ K
MAP captures both the rank and number of relevant items in the top-K predictions. It averages the precision at the rank of **each relevant item**, rewarding models that rank more relevant items higher.

For a given playlist $i$, let $\text{Rel}_i$ be the set of relevant (held-out) track indices, and let $\text{rank}_j$ be the rank of relevant item $j \in \text{Rel}_i$:

$$
AP_i@K = \frac{1}{|\text{Rel}_i \cap \text{TopK}_i|} \sum_{j \in \text{Rel}_i \cap \text{TopK}_i} \frac{\#\text{ relevant in top-}\text{rank}_j}{\text{rank}_j}
$$

Then, the overall MAP is:

$$
MAP@K = \frac{1}{N} \sum_{i=1}^{N} AP_i@K
$$

If a playlist has no relevant items in the top-K, \(AP_i@K = 0\).

### 3.1. Data Splitting & Metrics Functions
- For each playlist:
    - Randomly remove k tracks and save it as the test item.
	- Use the remaining tracks to construct your interaction matrix.

In [26]:
def leave_k_out_split(interaction_matrix, k_out=10):
    """
    For each playlist (row), randomly hide k_out tracks to serve as test items.
    
    Returns:
      - train_matrix: Interaction matrix with k_out tracks removed per playlist.
      - test_items: Dictionary mapping playlist index to list of held-out track indices.
    """
    train_matrix = interaction_matrix.copy()
    test_items = {}
    
    for i in range(train_matrix.shape[0]):
        present_indices = np.where(train_matrix[i] > 0)[0]
        if len(present_indices) <= k_out:
            continue  # Skip if not enough tracks to leave out
        test_indices = np.random.choice(present_indices, size=k_out, replace=False)
        train_matrix[i, test_indices] = 0  # Remove test items
        test_items[i] = test_indices.tolist()
    
    return train_matrix, test_items

def compute_metrics_for_playlist(predicted_scores, test_indices, k=10):
    """
    Compute Hit@K, MRR, and MAP@K for a single playlist over multiple held-out tracks.
    
    Parameters:
      - predicted_scores: 1D array of scores for all tracks.
      - test_indices: List of held-out (relevant) track indices.
      - k: Top K recommendations to consider.
      
    Returns:
      - hit: 1 if any test item is in top-k, else 0.
      - mrr: Reciprocal rank of the first relevant item in top-k (if any).
      - ap: Average Precision for all relevant items in top-k.
    """
    ranked_indices = np.argsort(-predicted_scores)
    top_k = ranked_indices[:k]

    hit = 1 if any(t in top_k for t in test_indices) else 0

    precisions = []
    num_hits = 0
    mrr = 0.0

    for rank_idx, track_idx in enumerate(ranked_indices[:k]):
        if track_idx in test_indices:
            num_hits += 1
            precision_at_k = num_hits / (rank_idx + 1)
            precisions.append(precision_at_k)
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)

    ap = np.mean(precisions) if precisions else 0.0

    return hit, mrr, ap

def evaluate_model(train_matrix, test_items, model_predict, k=10):
    """
    Generic evaluation function for recommendation models.
    
    Parameters:
      - train_matrix: Interaction matrix with held-out items removed.
      - test_items: Dictionary mapping playlist index to list of held-out track indices.
      - model_predict: Function that returns predicted scores for a given playlist.
      - k: Number of top recommendations to consider.
      
    Returns:
      - Average Hit Ratio, MRR, and MAP over all playlists.
    """
    hit_total = 0
    mrr_total = 0
    ap_total = 0
    num_playlists = len(test_items)
    
    for playlist_idx, test_indices in test_items.items():
        predicted_scores = model_predict(playlist_idx, train_matrix)
        hit, mrr, ap = compute_metrics_for_playlist(predicted_scores, test_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap
    
    hit_ratio = hit_total / num_playlists
    mrr_avg = mrr_total / num_playlists
    map_avg = ap_total / num_playlists
    return hit_ratio, mrr_avg, map_avg

# 4. Model Evaluation

- For each method, we create a wrapper that uses precomputed similarity matrices or latent factors, then evaluate using the generic function.

In [27]:
# Set K
K = 50

### 4.1. Evaluating Item-Based CF

In [28]:
# For reproducibility
np.random.seed(42)

# Create train-test split
train_matrix, test_items = leave_k_out_split(interaction_matrix)

# Precompute item similarity for efficiency
sim_matrix = compute_item_similarity(train_matrix)

# Define a wrapper for item-based prediction using the precomputed similarity matrix
def predict_item_based_wrapper(playlist_idx, train_matrix):
    return predict_item_based(playlist_idx, train_matrix, sim_matrix)

# Evaluate Item-Based CF
hit_ratio, mrr, map = evaluate_model(train_matrix, test_items, predict_item_based_wrapper, K)
print(f"Item-based CF - Hit Ratio @{K}:", hit_ratio)
print(f"Item-based CF - MRR @{K}:", mrr)
print(f"Item-based CF - MAP @{K}:", map)
print("Scores:", hit_ratio + mrr + map)

Item-based CF - Hit Ratio @50: 0.653
Item-based CF - MRR @50: 0.21881782413660797
Item-based CF - MAP @50: 0.15692573338486612
Scores: 1.028743557521474


### 4.2. Evaluating User-Based CF

In [29]:
# For reproducibility
np.random.seed(42)

# Create train-test split
train_matrix, test_items = leave_k_out_split(interaction_matrix)

# Precompute user similarity for efficiency
user_sim_matrix = compute_user_similarity(train_matrix)

# Define a wrapper for user-based prediction using the precomputed similarity matrix
def predict_user_based_wrapper(playlist_idx, train_matrix):
    return predict_user_based(playlist_idx, train_matrix, user_sim_matrix)

# Evaluate User-Based CF
hit_ratio, mrr, map = evaluate_model(train_matrix, test_items, predict_user_based_wrapper, k=10)
print(f"User-based CF - Hit Ratio @{K}:", hit_ratio)
print(f"User-based CF - MRR @{K}:", mrr)
print(f"User-based CF - MAP @{K}:", map)
print("Scores:", hit_ratio + mrr + map)

User-based CF - Hit Ratio @50: 0.365
User-based CF - MRR @50: 0.1793253968253968
User-based CF - MAP @50: 0.16760371693121687
Scores: 0.7119291137566137


### 4.3. Evaluating Matrix Factorization

#### 4.3.1 SVD

In [30]:
# For reproducibility
np.random.seed(42)

# (Re)create train-test split if needed
train_matrix, test_items = leave_k_out_split(interaction_matrix)

# Define a wrapper for matrix factorization prediction using latent factors P and Q
def predict_mf_wrapper(playlist_idx, train_matrix):
    return predict_mf(playlist_idx, train_matrix, P, Q)

# Evaluate Matrix Factorization
hit_ratio, mrr, map = evaluate_model(train_matrix, test_items, predict_mf_wrapper, k=10)
print(f"MF-SVD - Hit Ratio @{K}:", hit_ratio)
print(f"MF-SVD - MRR @{K}:", mrr)
print(f"MF-SVD - MAP @{K}:", map)
print("Scores:", hit_ratio + mrr + map)

MF-SVD - Hit Ratio @50: 0.688
MF-SVD - MRR @50: 0.44917857142857126
MF-SVD - MAP @50: 0.40392782690854134
Scores: 1.5411063983371125


#### 4.3.2 ALS

In [31]:
# For reproducibility
np.random.seed(42)

# (Re)create train-test split if needed
train_matrix, test_items = leave_k_out_split(interaction_matrix)

# Define a wrapper for ALS prediction using the latent factors P_als and Q_als
def predict_als_wrapper(playlist_idx, train_matrix):
    return predict_als(playlist_idx, train_matrix, P_als, Q_als)

# Evaluate the ALS model using our generic evaluation function
hit_ratio, mrr, map = evaluate_model(train_matrix, test_items, predict_als_wrapper, k=10)
print(f"MF-ALS - Hit Ratio @{K}:", hit_ratio)
print(f"MF-ALS - MRR @{K}:", mrr)
print(f"MF-ALS - MAP @{K}:", map)
print("Scores:", hit_ratio + mrr + map)

MF-ALS - Hit Ratio @50: 1.0
MF-ALS - MRR @50: 0.9351833333333336
MF-ALS - MAP @50: 0.9526786364638443
Scores: 2.887861969797178
